# Inversion run plots

All the plotting logic for the activation-inversion runs lives in this notebook: edit a cell to slice step ranges,
pick lines, or recolor, then re-run that cell.

**Data sources.** Nothing is imported from the repo. Every number comes from three files that `inversion/train.py`
writes into each run's directory, `RUNS_DIR/<run>/`:

| file | one row = |
|---|---|
| `config.json` | the run's settings (init, source, batch, LR, logging cadence, git commit) |
| `train_log.jsonl` | the **mean over `log_every` training steps** (`step` = the last of them): loss, token accuracy, and per sample source (`loss_s1`, `loss_s2`, …) which model's activations those samples came from |
| `val_log.jsonl` | **one val pass on one val source** (every `val_every` steps): loss, token accuracy, exact match, and the first 4 fixed val windows with the inverter's output |

Cells 2–4 print exactly which files were read and each run's settings.

In [ ]:
# Everything this notebook uses. Nothing comes from the repo (no `inversion.*` imports).
import json                      # parse config.json and the .jsonl logs
from pathlib import Path         # file paths

import matplotlib.pyplot as plt  # plotting
import pandas as pd              # tables, for slicing and filtering

pd.set_option("display.max_colwidth", None)

In [ ]:
# Where the runs live: relative to this notebook (inversion/), so ../runs/b128 is <repo>/runs/b128.
RUNS_DIR = Path("../runs/b128").resolve()
RUNS = ["s1_inv_s1_L6", "s1_inv_s2_L6", "rand_inv_s2_L6", "s1_inv_mixs12_L6", "rand_inv_mixs12_L6"]
print("RUNS_DIR =", RUNS_DIR)

In [ ]:
# Read the three files of every run. These dicts hold the raw rows; the tables below are built only from them.
def read_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f if line.strip()]

configs = {run: json.loads((RUNS_DIR / run / "config.json").read_text()) for run in RUNS}
raw_train = {run: read_jsonl(RUNS_DIR / run / "train_log.jsonl") for run in RUNS}
raw_val = {run: read_jsonl(RUNS_DIR / run / "val_log.jsonl") for run in RUNS}

# Which files were read, how many rows each has, and how far each run has got.
files = pd.DataFrame(
    [{"run": run, "file": str(RUNS_DIR / run / name), "rows": len(rows), "last_step": rows[-1]["step"]}
     for run in RUNS for name, rows in [("train_log.jsonl", raw_train[run]), ("val_log.jsonl", raw_val[run])]]
)
display(files)

# The settings each run was trained with (from its config.json).
SETTINGS = ["init", "source", "layer", "batch", "lr", "warmup", "log_every", "val_every", "n_val", "git_commit"]
settings = pd.DataFrame([{"run": run, **{k: configs[run].get(k) for k in SETTINGS}} for run in RUNS]).set_index("run")
display(settings)

In [ ]:
# Tidy tables: one row per line-point, so any slice is a plain pandas filter.

# train: one row per (run, step, sample_source). Mixed runs have two sample sources (s1 and s2) per log row.
rows = []
for run in RUNS:
    for r in raw_train[run]:
        for src in [k[len("loss_"):] for k in r if k.startswith("loss_")]:
            if r[f"n_{src}"] == 0:  # no samples of this source in these steps
                continue
            rows.append({"run": run, "init": configs[run]["init"], "trained_on": configs[run]["source"],
                         "step": r["step"], "sample_source": src, "loss": r[f"loss_{src}"],
                         "tok_acc": r[f"tok_acc_{src}"], "n": r[f"n_{src}"]})
train = pd.DataFrame(rows)

# val: one row per (run, step, val_source). Every run is evaluated on both s1 and s2, whatever it trained on.
val = pd.DataFrame(
    [{"run": run, "init": configs[run]["init"], "trained_on": configs[run]["source"], "step": r["step"],
      "val_source": r["source"], "loss": r["loss"], "tok_acc": r["tok_acc"], "exact_match": r["exact_match"]}
     for run in RUNS for r in raw_val[run]]
)

# val_examples: the same 4 fixed val windows at every val pass, with each run's output.
val_examples = pd.DataFrame(
    [{"run": run, "step": r["step"], "val_source": r["source"], "idx": ex["idx"], "n_correct": ex["n_correct"],
      "input_text": ex["input_text"], "output_text": ex["output_text"]}
     for run in RUNS for r in raw_val[run] for ex in r.get("examples", [])]
)

display(train.head(3), val.head(3), val_examples.head(3))

In [ ]:
# Style: edit here to recolor or relabel. Blue / orange / aqua-green are checked as a colorblind-safe set.
BLUE, ORANGE, GREEN = "#2a78d6", "#eb6834", "#1baf7a"
RED, YELLOW, PURPLE, NAVY = "#d62a2a", "#d69f2a", "#9f2ad6", "#2a4ed6"
INK, MUTED, GRID = "#2b2b29", "#6b6a64", "#e6e5e0"

# One entry per line: (run, source) -> color. "source" is the sample source for train plots, the val source for val.
COLOR = {
    ("rand_inv_s2_L6", "s2"): RED,
    ("s1_inv_s2_L6", "s2"): ORANGE,
    ("s1_inv_s1_L6", "s1"): BLUE,
    ("rand_inv_mixs12_L6", "s1"): BLUE,
    ("rand_inv_mixs12_L6", "s2"): ORANGE,
    ("s1_inv_mixs12_L6", "s2"): ORANGE,
    ("s1_inv_mixs12_L6", "s1"): BLUE,
}
LINESTYLE = {"random": "--", "s1": "-"}  # by inverter init: dashed random, solid s1-pretrained

# Legend labels for the train plots (val plots add " on <source>").
LABEL = {
    ("rand_inv_s2_L6", "s2"): "rand_inv_s2",
    ("s1_inv_s2_L6", "s2"): "s1_inv_s2",
    ("s1_inv_s1_L6", "s1"): "s1_inv_s1",
    ("rand_inv_mixs12_L6", "s1"): "rand_inv_mixs12 (s1 samples)",
    ("rand_inv_mixs12_L6", "s2"): "rand_inv_mixs12 (s2 samples)",
    ("s1_inv_mixs12_L6", "s2"): "s1_inv_mixs12 (s2 samples)",
    ("s1_inv_mixs12_L6", "s1"): "s1_inv_mixs12 (s1 samples)",
}

# Panels: title -> the (run, source) lines in it, in legend order.
PANELS = {
    "A)  single-source runs": [("rand_inv_s2_L6", "s2"), ("s1_inv_s2_L6", "s2"), ("s1_inv_s1_L6", "s1")],
    "B)  mixed runs (loss split by sample source)": [
        ("rand_inv_mixs12_L6", "s1"), ("rand_inv_mixs12_L6", "s2"),
        ("s1_inv_mixs12_L6", "s2"), ("s1_inv_mixs12_L6", "s1"),
    ],
}

In [ ]:
def plot_panels(df, y, panels, source_col, step_range=None, title="", ylabel="", labels=LABEL, ylim=None,
                figsize=(10, 9), ema_span=None):
    """One subplot per panel, stacked; each (run, source) is one line drawn through its logged points.

    df: `train` (source_col="sample_source") or `val` (source_col="val_source"); y: the column to plot.
    When `ema_span` is set, plot an EMA line and show the raw points as lighter dots.
    The y axis is linear and shared; by default it runs from 0 to just above the highest point drawn.
    """
    if step_range is not None:
        df = df[df.step.between(*step_range)]
    fig, axes = plt.subplots(len(panels), 1, figsize=figsize, sharex=True, sharey=True, squeeze=False)
    axes = axes[:, 0]
    for ax, (panel_title, lines) in zip(axes, panels.items()):
        for run, src in lines:
            d = df[(df.run == run) & (df[source_col] == src)].sort_values("step")
            style = LINESTYLE.get(configs[run]["init"], "-")
            color = COLOR[run, src]
            if ema_span is None:
                ax.plot(d.step, d[y], style, color=color, lw=1,
                        marker="o", ms=2.5, label=labels[run, src])
            else:
                ax.plot(d.step, d[y], linestyle="none", marker="o", ms=2.5,
                        color=color, alpha=0.15)
                ax.plot(d.step, d[y].ewm(span=ema_span, adjust=False).mean(), style,
                        color=color, lw=1.5, label=labels[run, src])
        ax.set_title(panel_title, loc="left", color=INK, fontsize=11)
        ax.set_ylabel(ylabel, color=INK)
        ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), frameon=False, fontsize=9, labelcolor=INK,
                  handlelength=3)
        ax.grid(True, color=GRID, lw=0.8)
        ax.spines[["top", "right"]].set_visible(False)
        ax.spines[["left", "bottom"]].set_color(MUTED)
        ax.tick_params(colors=MUTED)
    # Set the shared y range after every panel is drawn, so no panel clips another's lines.
    y_max = max((max(line.get_ydata(), default=0) for ax in axes for line in ax.get_lines()), default=1)
    axes[0].set_ylim(*(ylim or (0, y_max * 1.05)))
    if step_range is not None:
        axes[-1].set_xlim(*step_range)
    axes[-1].set_xlabel("training steps", color=INK)
    fig.suptitle(title, color=INK, fontsize=14)
    fig.tight_layout()
    return fig


# Shared settings for the titles (every run in RUNS_DIR was trained with the same ones).
LOG_EVERY, BATCH = settings.log_every.iloc[0], settings.batch.iloc[0]
VAL_EVERY, N_VAL = settings.val_every.iloc[0], settings.n_val.iloc[0]

## Train loss

In [ ]:
STEP_RANGE = (0, 2500)
fig = plot_panels(train, "loss", PANELS, "sample_source", STEP_RANGE,
                  title=f"Train loss, every {LOG_EVERY} steps. Batch size {BATCH}.", ylabel="cross-entropy (nats)",)
# fig.savefig(RUNS_DIR / "train_loss_AB_0-2500.png", dpi=150)

In [ ]:
STEP_RANGE = (1200, 10000)
fig = plot_panels(train, "loss", PANELS, "sample_source", STEP_RANGE,
                  title=f"Train loss, every {LOG_EVERY} steps. Batch size {BATCH}.", ylabel="cross-entropy (nats)", ema_span=50)
# fig.savefig(RUNS_DIR / "train_loss_AB_2500-5000.png", dpi=150)

## Validation

Same panels, but each line is the run evaluated on the val windows of the source it trained on. Every run is also
evaluated on the other seed (~13 nats, ~0 accuracy); to plot that, add e.g. `("s1_inv_s1_L6", "s2")` to a panel along
with a `COLOR` and `VAL_LABEL` entry.

In [ ]:
VAL_LABEL = {key: f"{LABEL[key].split(' (')[0]} on {key[1]}" for key in LABEL}
VAL_PANELS = {"A)  single-source runs": PANELS["A)  single-source runs"],
              "B)  mixed runs (evaluated on each source)": PANELS["B)  mixed runs (loss split by sample source)"]}
STEP_RANGE = None  # all of it; e.g. (0, 5000)
fig = plot_panels(val, "loss", VAL_PANELS, "val_source", STEP_RANGE, labels=VAL_LABEL,
                  title=f"Val loss, every {VAL_EVERY} steps, {N_VAL} windows per source.", ylabel="cross-entropy (nats)")
# fig.savefig(RUNS_DIR / "val_loss_AB.png", dpi=150)

In [ ]:
fig = plot_panels(val, "tok_acc", VAL_PANELS, "val_source", STEP_RANGE, labels=VAL_LABEL, ylim=(0, 1),
                  title=f"Val token accuracy, every {VAL_EVERY} steps, {N_VAL} windows per source.",
                  ylabel="token accuracy")
# fig.savefig(RUNS_DIR / "val_tok_acc_AB.png", dpi=150)

## Val examples

The first 4 val windows are the same at every val pass. This table shows each run's output for them at one step (the
latest step that every run has reached, unless you set `STEP`).

In [ ]:
common_steps = set.intersection(*(set(g.step) for _, g in val_examples.groupby("run")))
STEP = max(common_steps)
VAL_SOURCE = "s2"
ex = val_examples[(val_examples.step == STEP) & (val_examples.val_source == VAL_SOURCE)]
table = ex.pivot(index="idx", columns="run", values="output_text")[RUNS]
table.insert(0, "input", ex.groupby("idx").input_text.first())
print(f"step {STEP}, val source {VAL_SOURCE}")
display(table)

## 9-seed mix runs (batch 512)

Two runs whose training samples each come from **one of s1…s9, drawn uniformly at random**: a random-init inverter
and an s1-init one (otherwise identical: same batches, same draws). Same three files per run as above, in
`MIX_RUNS_DIR/<run>/`. Their `train_log.jsonl` rows carry `loss_s1 … loss_s9` (and `n_s1 … n_s9`), so any grouping
can be computed here; their `val_log.jsonl` rows are already lumped: the random-init run logs one group `s1-9`, the
s1-init run logs `s1` and `s2-9` (each val window's source drawn once from the group).

This section needs the cells above it (imports, `read_jsonl`, the style constants and `plot_panels`).

In [ ]:
MIX_RUNS_DIR = Path("../runs/b512").resolve()
MIX_RUNS = ["rand_inv_mixs1to9_L6", "s1_inv_mixs1to9_L6"]
print("MIX_RUNS_DIR =", MIX_RUNS_DIR)

mix_configs = {run: json.loads((MIX_RUNS_DIR / run / "config.json").read_text()) for run in MIX_RUNS}
mix_raw_train = {run: read_jsonl(MIX_RUNS_DIR / run / "train_log.jsonl") for run in MIX_RUNS}
mix_raw_val = {run: read_jsonl(MIX_RUNS_DIR / run / "val_log.jsonl") for run in MIX_RUNS}

display(pd.DataFrame(
    [{"run": run, "file": str(MIX_RUNS_DIR / run / name), "rows": len(rows), "last_step": rows[-1]["step"] if rows else None}
     for run in MIX_RUNS for name, rows in [("train_log.jsonl", mix_raw_train[run]), ("val_log.jsonl", mix_raw_val[run])]]
))
display(pd.DataFrame([{"run": run, **{k: mix_configs[run].get(k) for k in SETTINGS + ["val_groups"]}}
                      for run in MIX_RUNS]).set_index("run"))

In [ ]:
# Train loss per group of sample sources: the sample-weighted mean of loss_<s> over the group's members.
TRAIN_GROUPS = {"s1-9": [f"s{i}" for i in range(1, 10)], "s1": ["s1"], "s2-9": [f"s{i}" for i in range(2, 10)]}

rows = []
for run in MIX_RUNS:
    for r in mix_raw_train[run]:
        for group, members in TRAIN_GROUPS.items():
            n = sum(r[f"n_{m}"] for m in members)
            if n == 0:
                continue
            loss = sum(r[f"loss_{m}"] * r[f"n_{m}"] for m in members) / n
            acc = sum(r[f"tok_acc_{m}"] * r[f"n_{m}"] for m in members) / n
            rows.append({"run": run, "step": r["step"], "group": group, "loss": loss, "tok_acc": acc, "n": n})
mix_train = pd.DataFrame(rows)

# Val rows as logged: one per (run, step, group).
mix_val = pd.DataFrame(
    [{"run": run, "step": r["step"], "group": r["source"], "loss": r["loss"], "tok_acc": r["tok_acc"],
      "exact_match": r["exact_match"]} for run in MIX_RUNS for r in mix_raw_val[run]]
)
display(mix_train.head(3), mix_val.head(3))

In [ ]:
# Style for these runs: edit to recolor / regroup. plot_panels reads `configs` (for the init's line style) and `COLOR`.
configs.update(mix_configs)
COLOR.update({
    ("rand_inv_mixs1to9_L6", "s1-9"): GREEN,
    ("s1_inv_mixs1to9_L6", "s1"): BLUE,
    ("s1_inv_mixs1to9_L6", "s2-9"): ORANGE,
})
MIX_LABEL = {
    ("rand_inv_mixs1to9_L6", "s1-9"): "rand_inv_mixs1to9 (all of s1-s9)",
    ("s1_inv_mixs1to9_L6", "s1"): "s1_inv_mixs1to9 (s1)",
    ("s1_inv_mixs1to9_L6", "s2-9"): "s1_inv_mixs1to9 (s2-s9)",
}
MIX_PANELS = {
    "A)  random init: all sources lumped": [("rand_inv_mixs1to9_L6", "s1-9")],
    "B)  s1 init: s1 vs s2-s9": [("s1_inv_mixs1to9_L6", "s1"), ("s1_inv_mixs1to9_L6", "s2-9")],
}
MIX_LOG_EVERY, MIX_BATCH = mix_configs[MIX_RUNS[0]]["log_every"], mix_configs[MIX_RUNS[0]]["batch"]
MIX_VAL_EVERY, MIX_N_VAL = mix_configs[MIX_RUNS[0]]["val_every"], mix_configs[MIX_RUNS[0]]["n_val"]

In [ ]:
STEP_RANGE = None  # e.g. (0, 2500)
EMA_SPAN = None    # e.g. 50
fig = plot_panels(mix_train, "loss", MIX_PANELS, "group", STEP_RANGE, labels=MIX_LABEL, ema_span=EMA_SPAN,
                  title=f"Train loss, 9-seed mix, every {MIX_LOG_EVERY} steps. Batch size {MIX_BATCH}.",
                  ylabel="cross-entropy (nats)")
# fig.savefig(MIX_RUNS_DIR / "train_loss_mix9.png", dpi=150)

In [ ]:
fig = plot_panels(mix_val, "loss", MIX_PANELS, "group", STEP_RANGE, labels=MIX_LABEL,
                  title=f"Val loss, 9-seed mix, every {MIX_VAL_EVERY} steps, {MIX_N_VAL} windows per group.",
                  ylabel="cross-entropy (nats)")
# fig.savefig(MIX_RUNS_DIR / "val_loss_mix9.png", dpi=150)

## Batch-512 sweep: layers, and the layer-6 comparisons

All runs in `SWEEP_DIR/<run>/` use 10-token windows, batch 512 and 50k steps, and differ only in the flags in their
names. Every plot shows **val token accuracy on the inverted model's own val windows** (e.g. `s1_inv_s2` on s2;
the mixed runs on each of s1 and s2), against training steps on a log axis, so the fast start and the plateau near
1.0 are both visible. One step = 512 windows = 5,120 tokens.

1. **Layer sweep:** s1 inverting s1 at layers 3, 6, 9, 12 and `final` (after `final_layer_norm`). Left: the curves;
   right: steps to reach each threshold in `THRESHOLDS`, per layer, with each run's final accuracy under its layer.
   If the `s1_inv_s2_L*` sweep exists it gets a second row.
2. **Layer 6, single source:** `s1_inv_s1`, `s1_inv_s2`, `rand_inv_s2`.
3. **Layer 6, mixed source:** `s1_inv_mixs12`, `rand_inv_mixs12`, each on s1 and on s2.
4. **Layer 6, seed families:** `s1_inv_s2` (main seeds) vs `d1_inv_d2` (data seeds) vs `w1_inv_w2` (weight seeds).

This section needs the cells above it (imports, `read_jsonl`, `SETTINGS` and the style constants). Figures are saved
to `SWEEP_DIR/figures/`.

In [ ]:
SWEEP_DIR = Path("../runs/b512_sweep").resolve()
FIG_DIR = SWEEP_DIR / "figures"
print("SWEEP_DIR =", SWEEP_DIR)

sweep_configs, rows = {}, []
for run_dir in sorted(p for p in SWEEP_DIR.iterdir() if (p / "config.json").exists()):
    sweep_configs[run_dir.name] = json.loads((run_dir / "config.json").read_text())
    if (run_dir / "val_log.jsonl").exists():
        rows += [{"run": run_dir.name, "step": r["step"], "val_source": r["source"], "loss": r["loss"],
                  "tok_acc": r["tok_acc"], "exact_match": r["exact_match"]}
                 for r in read_jsonl(run_dir / "val_log.jsonl")]
sweep_val = pd.DataFrame(rows, columns=["run", "step", "val_source", "loss", "tok_acc", "exact_match"])

# Which runs were found, their settings, and how far each has got.
display(pd.DataFrame([{"run": run, **{k: c.get(k) for k in SETTINGS + ["n_tokens", "steps"]},
                       "last_val_step": sweep_val[sweep_val.run == run].step.max()}
                      for run, c in sweep_configs.items()]).set_index("run"))

In [ ]:
# Ordinal blue ramp for layers, light = early, dark = late (checked with the dataviz validator, --ordinal).
LAYER_COLORS = ["#86b6ef", "#5598e7", "#2a78d6", "#1c5cab", "#0d366b"]
SWEEP_LAYERS = ["3", "6", "9", "12", "final"]
SWEEP_ROWS = [("s1_inv_s1", "s1"), ("s1_inv_s2", "s2")]  # (run name without _L<layer>, val source); missing rows skipped
THRESHOLDS = (0.90, 0.98)  # "steps to reach" panel: one marker series per threshold
THRESHOLD_MARKERS = ["o", "D"]
FINAL_K = 5  # final accuracy = mean over the last FINAL_K val passes


def layer_label(layer):
    return "final" if layer == "final" else f"L{layer}"


def curve(run, src):
    return sweep_val[(sweep_val.run == run) & (sweep_val.val_source == src)].sort_values("step")


def first_step_at(d, threshold):
    hit = d[d.tok_acc >= threshold]
    return hit.step.iloc[0] if len(hit) else None


def style_ax(ax):
    ax.grid(True, color=GRID, lw=0.8)
    ax.spines[["top", "right"]].set_visible(False)
    ax.spines[["left", "bottom"]].set_color(MUTED)
    ax.tick_params(colors=MUTED)


def plot_acc(ax, lines, ylim=(0, 1.02)):
    """lines: (run, val source, label, color, linestyle). Val tok_acc vs step, log x, a dotted guide at 1.0."""
    for run, src, label, color, style in lines:
        d = curve(run, src)
        if len(d):
            ax.plot(d.step, d.tok_acc, style, color=color, lw=2, label=label)
    ax.axhline(1.0, color=MUTED, lw=1, ls=":")
    ax.set_xscale("log")
    ax.set_ylim(*ylim)
    batches = sorted({sweep_configs[run]["batch"] for run, *_ in lines if run in sweep_configs})
    ax.set_xlabel(f"training steps (batch {', '.join(map(str, batches))})", color=INK)
    ax.set_ylabel("val token accuracy", color=INK)
    ax.legend(loc="lower right", frameon=False, fontsize=9, labelcolor=INK, handlelength=3)
    style_ax(ax)

### 1. Layer sweep

In [ ]:
# A row needs at least two layers (s1_inv_s2_L6 alone comes from the layer-6 grid, not a sweep).
rows_present = [(base, src) for base, src in SWEEP_ROWS
                if sum(f"{base}_L{layer}" in sweep_configs for layer in SWEEP_LAYERS) >= 2]
fig, axes = plt.subplots(len(rows_present), 2, figsize=(13, 4.8 * len(rows_present)), squeeze=False,
                         gridspec_kw={"width_ratios": [2.2, 1]})
for (ax_curve, ax_steps), (base, src) in zip(axes, rows_present):
    layers = [layer for layer in SWEEP_LAYERS if f"{base}_L{layer}" in sweep_configs]
    color = dict(zip(SWEEP_LAYERS, LAYER_COLORS))
    plot_acc(ax_curve, [(f"{base}_L{layer}", src, f"layer {layer_label(layer)}", color[layer], "-")
                        for layer in layers])
    ax_curve.set_title(f"{base}: val token accuracy on {src}, by layer", loc="left", color=INK, fontsize=11)

    # Steps to reach each threshold, per layer. A threshold not reached yet is drawn hollow at the run's last step.
    x = range(len(layers))
    for threshold, marker in zip(THRESHOLDS, THRESHOLD_MARKERS):
        ys, reached = [], []
        for layer in layers:
            d = curve(f"{base}_L{layer}", src)
            step = first_step_at(d, threshold)
            ys.append(step if step is not None else (d.step.max() if len(d) else float("nan")))
            reached.append(step is not None)
        ax_steps.plot(x, ys, "-", color=MUTED, lw=1, zorder=1)
        for xi, yi, ok in zip(x, ys, reached):
            ax_steps.plot(xi, yi, marker, ms=8, color=INK, mfc=INK if ok else "white", mew=1.5, zorder=2)
        ax_steps.plot([], [], marker, ms=8, color=INK, ls="-", lw=1, label=f"{threshold:.0%}")
    ax_steps.plot([], [], "o", ms=8, color=INK, mfc="white", mew=1.5, ls="none", label="not reached (last step)")
    finals = [curve(f"{base}_L{layer}", src).tok_acc.tail(FINAL_K).mean() for layer in layers]
    ax_steps.set_xticks(list(x), [f"{layer_label(layer)}\n{acc:.1%}" for layer, acc in zip(layers, finals)])
    ax_steps.set_yscale("log")
    ax_steps.set_xlabel(f"layer  (final accuracy = mean of last {FINAL_K} val passes)", color=INK)
    ax_steps.set_ylabel("steps to reach val token accuracy", color=INK)
    ax_steps.set_title("time to threshold", loc="left", color=INK, fontsize=11)
    ax_steps.legend(loc="lower right", frameon=False, fontsize=9, labelcolor=INK)
    style_ax(ax_steps)
fig.suptitle("Later-layer activations take longer to invert", color=INK, fontsize=14)
fig.tight_layout()
FIG_DIR.mkdir(exist_ok=True)
fig.savefig(FIG_DIR / "layer_sweep.png", dpi=150)

# The same numbers as a table.
display(pd.DataFrame([{"run": f"{base}_L{layer}", "val_source": src,
                       **{f"steps_to_{t:.0%}": first_step_at(curve(f"{base}_L{layer}", src), t) for t in THRESHOLDS},
                       "final_acc": curve(f"{base}_L{layer}", src).tok_acc.tail(FINAL_K).mean(),
                       "last_step": curve(f"{base}_L{layer}", src).step.max()}
                      for base, src in rows_present for layer in SWEEP_LAYERS
                      if f"{base}_L{layer}" in sweep_configs]).set_index("run"))

### 2-4. Layer 6 comparisons

Line style = inverter init (solid: pretrained s1/d1/w1, dashed: random), as in the plots above.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
plot_acc(ax, [("s1_inv_s1_L6", "s1", "s1_inv_s1 on s1", BLUE, LINESTYLE["s1"]),
              ("s1_inv_s2_L6", "s2", "s1_inv_s2 on s2", ORANGE, LINESTYLE["s1"]),
              ("rand_inv_s2_L6", "s2", "rand_inv_s2 on s2", RED, LINESTYLE["random"])])
ax.set_title("Layer 6, single source: val token accuracy", loc="left", color=INK, fontsize=12)
fig.tight_layout()
fig.savefig(FIG_DIR / "L6_single.png", dpi=150)

In [ ]:
# Color = val source, line style = init.
fig, ax = plt.subplots(figsize=(9, 5))
plot_acc(ax, [("s1_inv_mixs12_L6", "s1", "s1_inv_mixs12 on s1", BLUE, LINESTYLE["s1"]),
              ("s1_inv_mixs12_L6", "s2", "s1_inv_mixs12 on s2", ORANGE, LINESTYLE["s1"]),
              ("rand_inv_mixs12_L6", "s1", "rand_inv_mixs12 on s1", BLUE, LINESTYLE["random"]),
              ("rand_inv_mixs12_L6", "s2", "rand_inv_mixs12 on s2", ORANGE, LINESTYLE["random"])])
ax.set_title("Layer 6, trained on a 50/50 mix of s1 and s2: val token accuracy per source", loc="left",
             color=INK, fontsize=12)
fig.tight_layout()
fig.savefig(FIG_DIR / "L6_mix.png", dpi=150)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
plot_acc(ax, [("s1_inv_s2_L6", "s2", "s1_inv_s2 on s2 (main seeds)", BLUE, "-"),
              ("d1_inv_d2_L6", "d2", "d1_inv_d2 on d2 (data seeds)", ORANGE, "-"),
              ("w1_inv_w2_L6", "w2", "w1_inv_w2 on w2 (weight seeds)", GREEN, "-")])
ax.set_title("Layer 6, inverting a different seed: val token accuracy", loc="left", color=INK, fontsize=12)
fig.tight_layout()
fig.savefig(FIG_DIR / "L6_families.png", dpi=150)